In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain.memory import ChatMessageHistory
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# CommaSeparatedListOutputParser
# 쉼표로 구분된 문자열을 파이썬 리스트로 변환

from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate

output_parser = CommaSeparatedListOutputParser()
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

# partial_variables를 사용하여 지시문을 프롬프트에 동적으로 삽입
prompt = PromptTemplate(
    template="인기 있는 과일 3가지를 나열해줘.\n{format_instructions}",
    input_variables=[],
    partial_variables={"format_instructions": format_instructions}
)

llm=ChatGoogleGenerativeAI(model="gemini-2.0-flash")

# 체인(Chain)
chain = prompt | llm | output_parser
# chain = prompt | llm

output = chain.invoke({})
print("언어 모델의 출력 (리스트):", output)

In [ ]:
# NumberedListOutputParser

# 언어 모델의 출력에서 숫자로 시작하는 라인을 찾아서 내용을 리스트로 변환

# 1. 대한민국 수도는 서울입니다.
# 2. 프랑스 수도는 파리입니다.
# 3. 일본 수도는 도쿄입니다.

# ['대한민국 수도는 서울입니다.', '프랑스 수도는 파리입니다.', '일본 수도는 도쿄입니다.']

In [ ]:
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import NumberedListOutputParser

output_parser = NumberedListOutputParser()
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

prompt = PromptTemplate(
    template="다음 주제에 대한 세 가지 아이디어를 번호가 매겨진 리스트로 생성해주세요:\n{topic}\n{format_instructions}",
    input_variables=["topic"],
    partial_variables={"format_instructions": format_instructions},
)

llm=ChatGoogleGenerativeAI(model="gemini-2.0-flash")
chain = prompt | llm | output_parser
# chain = prompt | llm

topic = "새로운 소셜 미디어 앱 아이디어"
output = chain.invoke({"topic": topic})

# 결과 확인
print('-'*50)
print(output)
print('-'*50)
for x in output:
    print(x)

In [ ]:
from enum import Enum

# Enum 클래스을 정의
# 멤버는 이름이 있고 값이 있는 상수 집합
# 이름은 SPORTS, 값은 "스포츠", 접근은 Topic.SPORTS

class Topic(Enum):
    SPORTS = "스포츠"
    SCIENCE = "과학"
    ENTERTAINMENT = "연예"
    TECHNOLOGY = "기술"
    NEWS = "뉴스"

In [ ]:
# EnumOutputParser
# 언어 모델이 생성한 텍스트를 Enum의 멤버로 변환

from langchain.prompts import PromptTemplate
from langchain.output_parsers.enum import EnumOutputParser
from langchain_openai import ChatOpenAI
from enum import Enum

class Topic(Enum):
    SPORTS = "스포츠"
    SCIENCE = "과학"
    ENTERTAINMENT = "연예"
    TECHNOLOGY = "기술"
    NEWS = "뉴스"

output_parser = EnumOutputParser(enum=Topic)
# 언어 모델이 생성한 문자열 출력을 Topic이라는 열거형(Enum) 클래스의 멤버로 변환

format_instructions = output_parser.get_format_instructions()
print(format_instructions)

prompt = PromptTemplate(
    template="다음 텍스트의 주제를 다음 옵션에 해당하는 단어로만 출력: {enum_options}\n\n텍스트: {text}",
    input_variables=["text"],
    partial_variables={"enum_options": output_parser.get_format_instructions()},
)

llm=ChatGoogleGenerativeAI(model="gemini-2.0-flash")
chain = prompt | llm | output_parser

input_text = "NASA, 인공위성을 이용하여 화성 탐사 계획을 발표했다."
output = chain.invoke({"text": input_text})

print(output.name)
print(output.value) # 할당된 값

In [ ]:
# DatetimeOutputParser
# 언어 모델이 생성한 날짜와 시간 정보를 애플리케이션에서 바로 사용할 수 있는 타입으로 변환

from langchain_core.prompts import PromptTemplate
from langchain.output_parsers import DatetimeOutputParser

output_parser = DatetimeOutputParser()

format_instructions = output_parser.get_format_instructions()
print(format_instructions)

prompt = PromptTemplate(
    template="2025년 12월 25일 크리스마스 날짜와 시간을 말해줘. 형식은 다음과 같아.\n{format_instructions}",
    input_variables=[],
    partial_variables={"format_instructions": format_instructions},
)

llm=ChatGoogleGenerativeAI(model="gemini-2.0-flash")
chain = prompt | llm | output_parser

output = chain.invoke({})

print('-'*50)
print("파싱된 객체:", output)
print("객체 타입:", type(output))

print("년도:", output.year)
print("월:", output.month)
print("일:", output.day)

In [ ]:
# RetryWithErrorOutputParser, 특정 파서

# 1. 특정 파서가 자신의 역할을 수행하려고 함
# 2. 파싱이 실패하면 ValidationError가 발생 >> 실패 이유는 LLM
# 3. RetryWithErrorOutputParser는 원래 프롬프트, 실패한 응답, 오류 메시지를 LLM에게 전달
# 4. LLM은 오류를 참고하여 응답을 다시 만듬
# 5. 단계 1부터 실행, 정해진 횟수만큼 반복

In [ ]:
# RetryWithErrorOutputParser - 1

from langchain.output_parsers import RetryWithErrorOutputParser, PydanticOutputParser
from langchain.prompts import PromptTemplate
from langchain.schema import PromptValue
from pydantic import BaseModel, Field

# 프롬프트 템플릿 정의
prompt = PromptTemplate.from_template(
    "다음 정보를 JSON 형식으로 제공해주세요: 이름과 나이.\n지민이는 나이가 28입니다."
)

llm=ChatGoogleGenerativeAI(model="gemini-2.0-flash")

chain = prompt | llm
response = chain.invoke({})
completion = response.content

print(completion)
# ---------------------- 여기까지는 파서는 아직 적용되지 않은 시점 ----------------------

# 아래 셀에서 계속

In [ ]:

class Person(BaseModel):
    name: str = Field(..., description="이름")
    age: int = Field(..., description="나이")

parser = PydanticOutputParser(pydantic_object=Person)
# LLM의 출력을 Person Pydantic 모델의 인스턴스
# name과 age라는 두 개의 필드 형식으로 응답하게끔 유도

# LLM과 파서를 결합하여 재시도 로직을 준비하는 단계
# "특정 언어모델에 대해서 특정 파서를 적용하겠다는 의미"
retry_parser = RetryWithErrorOutputParser.from_llm(
    llm=llm,
    # llm을 밝히는 이유는 오류를 수정하기 위해 새로운 응답을 생성해야 하기 때문
    
    parser=parser,
    # 나중에 전달받을 completion에 대해서 적용할 파서

    # max_retries=3 
    # # 실제 시도가 아니라 준비
)

parsed_result = retry_parser.parse_with_prompt(completion, prompt.format_prompt())
# completion을 얻는데 사용한 prompt을 다시 전달하는 이유는 맥락 제공임
# prompt.format_prompt(): PromptTemplate 객체에 변수 값을 채워 넣어 완성된 PromptValue 객체를 반환
# {변수}가 포함된 문자열 > PromptTemplate > 값이 채워짐 > PromptValue
# prompt | llm에서 prompt는 PromptValue 객체 형태로 변환되어서 넘김

# parse_with_prompt(): LLM 출력과 함께 원래 사용한 프롬프트도 전달
# parse(): 단순히 LLM 출력 문자열을 전달 > 원래 어떤 프롬프트로 LLM을 호출했는지 몰라도 충분한 경우

# 1. retry_parser는 parser를 사용해서 completion을 파싱 시도
# 2. 이 과정에서 오류가 발생하면 retry_parser는 오류를 감지하고
# 오류가 발생하지 않으면 모델의 출력 completion이 파싱 과정을 거쳐 parsed_result가 됨
# completion은 단순 문자열이며 parsed_result는 파이썬 객체라는 차이가 있음

# 3. 처음 사용했던 프롬프트 + 오류가 발생한 completion을 결합해서 LLm에게 새로운 프롬프트를 보냄
# 다음의 내용으로 프롬프트를 구성

# "당신은 이전에 {원본 프롬프트}에 대해 {오류가 난 응답}을 보냈습니다."
# "하지만 이 응답은 {오류 메시지} 때문에 유효하지 않습니다."
# "올바른 형식은 {원하는 출력 형식 지침}입니다."
# "이전 오류를 바탕으로 올바른 형식으로 다시 작성해주세요."

print(parsed_result)


In [ ]:
# 1.
# chain = prompt | llm
# response = chain.invoke({})
# completion = response.content

# 2.
# parser = PydanticOutputParser(pydantic_object=Person)
# completion | parser  -----------> XXX

# 3. 준비 단계; 잘못되면 물어볼 대상, 파서는 누구인가?
# retry_parser = RetryWithErrorOutputParser.from_llm(
#     llm=llm,
#     parser=parser
# )

# 4. 실질적인 실행 (파싱)
# parsed_result = retry_parser.parse_with_prompt(completion, prompt.format_prompt())
# parser가 completion을 파싱 >> 성공 아니면 실패
# 실패하면 다시 LLM에게 요청: 원래 사용했던 프롬프트, 잘못된 결과물(completion), 에러 메시지 | llm
# 성공하면 completion >>> 단순 문자열에서 (parser에서 요구하는) 타입으로 변환(retry_parser)


In [ ]:
# RetryWithErrorOutputParser - 2

import json
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain.output_parsers import RetryWithErrorOutputParser
from pydantic import BaseModel, Field
from langchain_core.messages import AIMessage

class UserInfo(BaseModel):
    name: str = Field(description="사용자의 이름")
    age: int = Field(description="사용자의 나이")

main_parser = PydanticOutputParser(pydantic_object=UserInfo)

# 파서의 기대: {"name": "이정후", "age": 25}

# 오류를 유도하는 프롬프트 템플릿 생성
# 의도적으로 JSON 앞에 '결과: ' 텍스트를 추가하도록 지시
prompt_template = PromptTemplate(
    template="한 사람의 이름과 나이를 JSON 형식으로 알려줘. \n"
             "주의: JSON 앞에 '결과: ' 텍스트를 반드시 추가해.\n"
             "{format_instructions}\n"
             "사용자: 이정후, 25세",
    input_variables=[],
    partial_variables={"format_instructions": main_parser.get_format_instructions()}
)

llm=ChatGoogleGenerativeAI(model="gemini-2.0-flash")
chain = prompt_template | llm
response = chain.invoke({})
completion = response.content

print("--- 1단계: LLM에서 반환된 원본 객체 ---")
print("객체 타입:", type(response))
print(completion)

# 아래 셀에서 계속

In [ ]:
from langchain_openai import ChatOpenAI

fixer_parser = RetryWithErrorOutputParser.from_llm(
    parser=main_parser,
    # llm=ChatGoogleGenerativeAI(model="gemini-2.0-flash"),   # 오류 수정 안됨
    llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash"),
    # llm = ChatOpenAI(model="gpt-4o-mini"), # 오류 수정 안됨
    max_retries=3
)

final_output = None
try:
    formatted_prompt = prompt_template.format_prompt(text_input=None)
    final_output = fixer_parser.parse_with_prompt(completion, formatted_prompt)

    print("\n--- 2단계: 오류 수정 후 최종 결과 ---")
    print(final_output)

except Exception as e:
    print(final_output)
    print(f"\n오류 복구 실패: {e}")

In [ ]:
# RegexParser
# LLM의 출력에서 정규식(regex) 패턴을 추출하는 데 사용되는 파서

In [ ]:
# RegexParser

from langchain_core.prompts import PromptTemplate
from langchain.output_parsers.regex import RegexParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import AIMessage
import re

# regex: 정규식(Regular Expression) 패턴을 정의

parser = RegexParser(
    regex=r"(\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b)",
    
    # \b는 단어의 경계(문자열의 시작, 문자열의 끝, 공백이나 .,? 등)
    # \bcat\b
    # "the cat sat", "a cat, you know"

    # [A-Za-z0-9._%+-]+   A, Aa, A_a, A%
    # A-Za-z0-9: 모든 영어 대소문자와 숫자
    # ._%+-: 마침표, 밑줄, 퍼센트, 더하기, 빼기 기호
    # []+: []내의 문자가 한 번 이상 반복

    # @ 문자와 정확히 일치
    # \.: 마침표 문자
    
    # [A-Z|a-z]{2,}
    # A-Z: 영어 대문자
    # a-z: 영어 소문자
    # {2,}: 최소 2번 이상 반복

    output_keys=["email"]
    # output_keys: 추출된 값에 할당할 키 이름
    # >>> 딕셔너리 출력을 의미
)

# 프롬프트 템플릿
prompt = PromptTemplate.from_template(
    "사용자에게서 받은 정보를 요약하여 이메일 주소를 알려줘.\n"
    "사용자 정보: 제 이름은 김지민이고, 연락처는 jimin.kim@example.com 입니다."
)

llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# llm의 응답(AIMessage)에서 문자열에 해당하는 .content 속성만 전달
chain = prompt | llm | RunnableLambda(lambda x: x.content) | parser

output = chain.invoke({})

print(output, type(output))

In [ ]:
# OutputFixingParser
# LangChain에서 LLM의 출력이 예상한 형식과 다를 때 자동으로 수정
# PydanticOutputParser와 함께 사용

# PydanticOutputParser >> 어떤 필드가 있고, 그 필드의 타입, 필수

# RetryWithErrorOutputParser는 일반적(범용)

In [ ]:
# OutputFixingParser - 1

from langchain.output_parsers import PydanticOutputParser, OutputFixingParser
from langchain.schema import BaseOutputParser
from pydantic import BaseModel, Field

# 출력 형식을 정의할 Pydantic 모델
class Person(BaseModel):
    name: str = Field(..., description="The person's name")
    age: int = Field(..., description="The person's age")

base_parser = PydanticOutputParser(pydantic_object=Person)

llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

fixing_parser = OutputFixingParser.from_llm(parser=base_parser, llm=llm) # 실제 시도가 아니라 준비

# 잘못된 형식의 출력 예시
bad_output = "Name: Alice\nAge: twenty five"

# 자동 수정 및 파싱
parsed = fixing_parser.parse(bad_output)
# parse_with_prompt(): LLM 출력과 함께 원래 사용한 프롬프트도 전달
# parse(): 단순히 LLM 출력 문자열을 전달 > 원래 어떤 프롬프트로 LLM을 호출했는지 몰라도 충분한 경우

print(parsed)


In [ ]:
# OutputFixingParser - 2

from langchain.output_parsers import PydanticOutputParser, OutputFixingParser
from langchain.prompts import PromptTemplate
from langchain.schema import PromptValue
from pydantic import BaseModel, Field

class Person(BaseModel):
    name: str = Field(..., description="The person's name")
    age: int = Field(..., description="The person's age")

base_parser = PydanticOutputParser(pydantic_object=Person)

llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

fixing_parser = OutputFixingParser.from_llm(parser=base_parser, llm=llm)

prompt_template = PromptTemplate.from_template(
    "Extract the person's name and age from the following text and return as JSON:\n{text}"
)
prompt_value: PromptValue = prompt_template.format_prompt(text="Alice is twenty five years old.")

# LLM 출력 (형식이 틀린 예시)
# LLM의 결과를 사용하지 않는 이유
# 1. 어떻게 동작하는지 보여주기 위함
# 2. 테스트와 디버깅
# 3. LLM의 출력이 예측 불가능

bad_output = "Name: Alice\nAge: twenty five"

# parse_with_prompt()로 수정 및 파싱
parsed = fixing_parser.parse_with_prompt(bad_output, prompt_value)
# 1. fixing_parser는 내부적으로 base_parser.parse(bad_output)을 먼저 시도
#    >>> bad_output을 Person 모델에 맞게 파싱 시도
# 2. bad_output = "Name: Alice\nAge: twenty five"는 파싱 실패
# 3. OutputFixingParser는 프롬프트를 수정하는 단계를 진행

# 당신은 다음 프롬프트에 응답했습니다:
# "Extract the person's name and age from the following text and return as JSON:
# Alice is twenty five years old."

# 당신의 응답은 다음과 같았습니다:
# "Name: Alice\nAge: twenty five"

# 하지만 이 응답은 다음 오류 때문에 유효하지 않습니다:
# "ValidationError: age must be an integer"

# 올바른 출력 형식은 다음과 같습니다:
# {name: string, age: integer} in JSON format

# 이전 오류를 바탕으로 올바른 형식으로 다시 작성해주세요.

# 4. 위에서 생성한 프롬프트를 llm에게 전달하여 출력을 요청

# parse_with_prompt(): LLM 출력과 함께 원래 사용한 프롬프트도 전달
# parse(): 단순히 LLM 출력 문자열을 전달 > 원래 어떤 프롬프트로 LLM을 호출했는지 몰라도 충분한 경우

print(parsed)


In [ ]:
# PandasDataFrameOutputParser

In [ ]:
# JSON 객체는 {}로 둘러싸인 키-값의 집합

# {
#   "이름": "홍길동",
#   "나이": 30,
#   "결혼여부": false
# }

In [ ]:
# PandasDataFrameOutputParser - 1

from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
import json

llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# JSON 객체 예시를 제공해서 LLM이 그 구조를 따르게 함
# 문자열의 경우
# {{ > {
# }} > }
# 변수의 경우에는 {} 그대로 사용
prompt = PromptTemplate.from_template(
    """다음 정보를 JSON 객체로 제공해주세요. 각 키의 값은 배열이어야 합니다. 오직 JSON 객체만 반환하고, 다른 텍스트는 포함하지 마세요.

예시:
{{
  "이름": ["홍길동", "김철수"],
  "나이": [25, 30]
}}

작업:
{input}
"""
)

chain = prompt | llm

input_info = "이름은 박영희이고 나이는 45, 그리고 이순신은 50세야."
response = chain.invoke({"input": input_info})

print("--- LLM이 생성한 JSON ---")
print(response.content)

# 다음 셀에서 계속

In [ ]:

# 5. 파이썬 객체로 변환
try:
    parsed_output = json.loads(response.content.strip("```json\n").strip("```"))
    # json.loads()는 JSON 같이 보이는 문자열을 파이썬 객체(딕셔너리)로 변환
    print("\n--- 파싱된 파이썬 객체 ---")
    print(parsed_output)
    print(f"타입: {type(parsed_output)}")
except json.JSONDecodeError as e:
    print(f"\n파싱 오류: {e}")

In [ ]:
# PandasDataFrameOutputParser - 2

import pandas as pd
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
import json

# JSON 객체 배열
prompt = PromptTemplate.from_template(
    """다음 정보를 JSON 배열로 제공해주세요. 오직 JSON 배열만 반환하고, 다른 텍스트는 포함하지 마세요.

예시:
입력: 이름은 홍길동이고 나이는 25, 그리고 김철수는 30세야.
출력:
[
  {{ "이름": "홍길동", "나이": 25 }},
  {{ "이름": "김철수", "나이": 30 }}
]

작업:
입력: {input}
출력:
"""
)

llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# 모델은 다음과 같이 출력
# ```json
# JSON 내용
# ```
def clean_llm_output(message: AIMessage) -> str:
    text = message.content

    if text.startswith('```json') and text.endswith('```'):
        return text.removeprefix('```json\n').removesuffix('```').strip()
    return text.strip()

# 체인 구성: LLM에서 JSON 객체를 받음
json_chain = prompt | llm | RunnableLambda(clean_llm_output)
input_data = "지민이는 28세이고 95점, 태형이는 29세이고 88점이야."
json_output = json_chain.invoke({"input": input_data})

print(json_output)

# 아래 셀에서 계속


In [ ]:
print(type(json_output))
print(type(json.loads(json_output)))

# 아래 셀에서 계속

In [ ]:
# JSON 객체 배열 >>> DataFrame으로 변환
def parse_to_dataframe(json_string):
    try:
        data_list = json.loads(json_string)
        return pd.DataFrame(data_list)
    except Exception as e:
        raise ValueError(f"JSON을 DataFrame으로 변환하는 데 실패했습니다: {e}")

df = parse_to_dataframe(json_output)
df